# 05 - Meta-Controller: Selecting Worker-Agent Training Context

This notebook implements the workshop experiment. A controller chooses between two worker training policies after every stage:

- **A**: stripped action-observation context.
- **B**: retained action-observation context.

API families are split into disjoint fit, controller-validation, and final-test sets. The controller sees validation metrics only. Final results are computed only on held-out API families and stream orders. The controller is deterministic and nonparametric: it selects the candidate with the highest predefined reliability-aware validation score.

In [ ]:
# Install in a fresh Colab runtime.
!pip install -q transformers accelerate peft bitsandbytes trl huggingface_hub tqdm

In [ ]:
import gc
import hashlib
import json
import os
import pickle
import random
import shutil
import time
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer,
)

SEED = 42
MAX_SEQ_LEN = 1024
EVAL_MAX_SAMPLES = 64
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
BASE_EPOCHS = 3
LR = 2e-4
TOKEN_MATCHED = True
EXPERIMENT_DIR = Path('meta_controller_seed42')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), 'This notebook requires a GPU runtime.'
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Run 01_data_prep.ipynb first after pulling the updated notebook.
with open('preprocessed_data/preprocessed.pkl', 'rb') as f:
    data = pickle.load(f)

blocks = data['blocks']
prep_config = data['config']
MODEL_NAME = prep_config['model_name']
NUM_BLOCKS = len(blocks)
SYSTEM_PROMPT = prep_config['system_prompt']
print(f'Model: {MODEL_NAME}; blocks: {NUM_BLOCKS}')
assert all('train_entries_raw' in b for b in blocks), (
    'Re-run 01_data_prep.ipynb: raw training entries are required.'
)

## 1. Disjoint API-Family Partitions

The split is by API family, never by individual example. A family is assigned deterministically to fit, controller validation, or final test. This prevents the controller from observing examples from a family that appears in the final test set.

In [ ]:
def family_partition(api_name, seed=SEED):
    # Stable across Python processes and independent of dictionary ordering.
    digest = hashlib.sha256(f'{seed}:{api_name}'.encode('utf-8')).hexdigest()
    bucket = int(digest[:8], 16) % 10
    if bucket < 6:
        return 'fit'
    if bucket < 8:
        return 'validation'
    return 'test'

partitions = {}
for block in blocks:
    # Fit uses only the original training split. Validation and test use
    # only the original held-out split, with API families kept disjoint.
    by_partition = {'fit': [], 'validation': [], 'test': []}
    for entry in block['train_entries_raw']:
        if family_partition(entry['api_name']) == 'fit':
            by_partition['fit'].append(entry)
    for entry in block['eval_entries_raw']:
        role = family_partition(entry['api_name'])
        if role in ('validation', 'test'):
            by_partition[role].append(entry)
    partitions[block['block_id']] = by_partition
    families = {
        role: sorted({e['api_name'] for e in entries})
        for role, entries in by_partition.items()
    }
    assert not (set(families['fit']) & set(families['validation']))
    assert not (set(families['fit']) & set(families['test']))
    assert not (set(families['validation']) & set(families['test']))
    print('D{}: '.format(block['block_id']) + ', '.join(
        f'{role}={len(families[role])} families/{len(by_partition[role])} examples'
        for role in ('fit', 'validation', 'test')
    ))

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
with open(EXPERIMENT_DIR / 'split_manifest.json', 'w') as f:
    json.dump({
        str(bid): {role: sorted({e['api_name'] for e in entries})
                  for role, entries in by_partition.items()}
        for bid, by_partition in partitions.items()
    }, f, indent=2)

In [ ]:
# Tokenizer is initialized before formatting/token matching cells run.
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def strip_trajectory_lines(text):
    lines = []
    for line in text.split('\n'):
        s = line.strip()
        if s.startswith('API-Request:') or s.startswith('API-Response:'):
            continue
        if 'Received API Response' in line or 'Generate API Request' in line:
            continue
        lines.append(line)
    return '\n'.join(lines).strip()

def format_entry(entry, condition):
    context = strip_trajectory_lines(entry['input']) if condition == 'A' else entry['input']
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': context},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    response = f"{entry.get('output', '')}{tokenizer.eos_token}"
    return prompt + response, len(tokenizer(prompt, add_special_tokens=False)['input_ids'])

def stable_order(entries, condition):
    # Same example order for A/B; selection is not affected by a random
    # shuffle that differs between candidate policies.
    keyed = []
    for entry in entries:
        key = hashlib.sha256(
            '{}:{}:{}'.format(SEED, entry['api_name'], entry['input']).encode('utf-8')
        ).hexdigest()
        keyed.append((key, entry))
    return [entry for _, entry in sorted(keyed)]

def prepare_training_entries(entries):
    # Equalize observed training tokens by capping B to A's token budget.
    # This is a transparent cost control, not a claim that the samples are
    # identical: B may use fewer examples because each example is longer.
    ordered = {cond: stable_order(entries, cond) for cond in ('A', 'B')}
    formatted = {
        cond: [format_entry(e, cond) for e in ordered[cond]]
        for cond in ('A', 'B')
    }
    target = sum(
        min(len(tokenizer.encode(text, add_special_tokens=False)), MAX_SEQ_LEN)
        for text, _ in formatted['A']
    )
    if not TOKEN_MATCHED:
        target = sum(
            min(len(tokenizer.encode(text, add_special_tokens=False)), MAX_SEQ_LEN)
            for text, _ in formatted['B']
        )
    selected = []
    total = 0
    for entry, (text, plen) in zip(ordered['B'], formatted['B']):
        n_tokens = min(len(tokenizer.encode(text, add_special_tokens=False)), MAX_SEQ_LEN)
        if selected and total + n_tokens > target:
            break
        selected.append((entry, text, plen))
        total += n_tokens
    if TOKEN_MATCHED:
        b_entries = [entry for entry, _, _ in selected]
        b_formatted = [(text, plen) for _, text, plen in selected]
    else:
        b_entries, b_formatted = ordered['B'], formatted['B']
    return {
        'A': (ordered['A'], formatted['A']),
        'B': (b_entries, b_formatted),
    }

## 2. Worker Model and Training Utilities

In [ ]:
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def load_base_worker():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, token=HF_TOKEN, quantization_config=bnb_config,
        device_map='auto', torch_dtype=torch.bfloat16, attn_implementation='sdpa'
    )
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(
        r=32, lora_alpha=64,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'
    ))
    return model

def load_worker(adapter_path=None):
    if adapter_path is None:
        return load_base_worker()
    # Load a fresh quantized base, then attach only the selected adapter.
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, token=HF_TOKEN, quantization_config=bnb_config,
        device_map='auto', torch_dtype=torch.bfloat16, attn_implementation='sdpa'
    )
    base = prepare_model_for_kbit_training(base)
    return PeftModel.from_pretrained(base, adapter_path, is_trainable=True)

class TextDataset(Dataset):
    def __init__(self, formatted, tokenizer, max_length):
        self.items = []
        for text, prompt_len in formatted:
            enc = tokenizer(text, truncation=True, max_length=max_length,
                            add_special_tokens=False)
            labels = list(enc['input_ids'])
            labels[:min(prompt_len, len(labels))] = [-100] * min(prompt_len, len(labels))
            self.items.append({
                'input_ids': enc['input_ids'],
                'attention_mask': enc['attention_mask'],
                'labels': labels,
            })
    def __len__(self):
        return len(self.items)
    def __getitem__(self, index):
        return self.items[index]

class CausalLMCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, features):
        width = max(len(f['input_ids']) for f in features)
        pad = self.tokenizer.pad_token_id
        return {
            'input_ids': torch.tensor([f['input_ids'] + [pad] * (width - len(f['input_ids'])) for f in features]),
            'attention_mask': torch.tensor([f['attention_mask'] + [0] * (width - len(f['attention_mask'])) for f in features]),
            'labels': torch.tensor([f['labels'] + [-100] * (width - len(f['labels'])) for f in features]),
        }

def train_worker(model, formatted, stage_name):
    dataset = TextDataset(formatted, tokenizer, MAX_SEQ_LEN)
    args = TrainingArguments(
        output_dir=f'/tmp/meta_controller_{stage_name}',
        num_train_epochs=BASE_EPOCHS, per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS, learning_rate=LR,
        bf16=True, logging_steps=10, save_strategy='no', report_to='none',
        optim='paged_adamw_8bit', warmup_ratio=0.1, lr_scheduler_type='cosine',
        seed=SEED, dataloader_pin_memory=True, dataloader_num_workers=2,
        gradient_checkpointing=True
    )
    result = Trainer(
        model=model, train_dataset=dataset, args=args,
        data_collator=CausalLMCollator(tokenizer),
    ).train()
    return float(result.training_loss)


## 3. Validation and Final-Test Metrics

In [ ]:
CALL_RE = __import__('re').compile(r'\[\s*([A-Za-z_][A-Za-z0-9_]*)\((.*?)\)\s*\]', __import__('re').DOTALL)
PARAM_RE = __import__('re').compile(r"(\w+)='([^']*)'")

def parse_api_call(text):
    match = CALL_RE.search(text)
    if not match:
        return None, None
    return match.group(1), {k: v for k, v in PARAM_RE.findall(match.group(2))}

def normalize_params(params):
    return {k.strip().lower(): v.strip().lower() for k, v in params.items()}

def make_subset(entries, seed):
    scored = [e for e in entries if parse_api_call(e.get('output', ''))[0] is not None]
    if len(scored) <= EVAL_MAX_SAMPLES:
        return scored
    rng = random.Random(seed)
    return [scored[i] for i in sorted(rng.sample(range(len(scored)), EVAL_MAX_SAMPLES))]

def build_generation_prompt(entry, condition):
    context = strip_trajectory_lines(entry['input']) if condition == 'A' else entry['input']
    return tokenizer.apply_chat_template([
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': context},
    ], tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def evaluate_entries(model, entries, condition, sample_seed):
    model.eval()
    entries = make_subset(entries, sample_seed)
    counts = {'total': len(entries), 'exact_full': 0, 'name': 0,
              'valid': 0, 'malformed_or_no_call': 0, 'wrong_api': 0,
              'wrong_params': 0}
    for entry in entries:
        expected_api, expected_params = parse_api_call(entry.get('output', ''))
        expected_params = normalize_params(expected_params)
        prompt = build_generation_prompt(entry, condition)
        enc = tokenizer(prompt, truncation=True, max_length=MAX_SEQ_LEN - 128,
                        return_tensors='pt').to(model.device)
        generated = model.generate(**enc, max_new_tokens=128, do_sample=False,
                                   pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(generated[0][enc['input_ids'].shape[1]:],
                                skip_special_tokens=True)
        predicted_api, predicted_params = parse_api_call(text)
        if predicted_api is None:
            counts['malformed_or_no_call'] += 1
            continue
        counts['valid'] += 1
        if predicted_api.lower() != expected_api.lower():
            counts['wrong_api'] += 1
            continue
        counts['name'] += 1
        if normalize_params(predicted_params) == expected_params:
            counts['exact_full'] += 1
        else:
            counts['wrong_params'] += 1
    total = counts['total'] or 1
    counts.update({
        'exact_acc': counts['exact_full'] / total,
        'name_acc': counts['name'] / total,
        'valid_rate': counts['valid'] / total,
        'malformed_rate': counts['malformed_or_no_call'] / total,
        'wrong_api_rate': counts['wrong_api'] / total,
    })
    model.train()
    return counts

def aggregate_metrics(metrics_by_block):
    if not metrics_by_block:
        return {'exact_acc': 0.0, 'name_acc': 0.0, 'valid_rate': 0.0,
                'malformed_rate': 0.0, 'wrong_api_rate': 0.0}
    keys = ('exact_acc', 'name_acc', 'valid_rate', 'malformed_rate', 'wrong_api_rate')
    return {key: float(np.mean([m[key] for m in metrics_by_block.values()])) for key in keys}

def controller_score(aggregate):
    # Frozen before looking at final-test results. Higher is better.
    return (0.50 * aggregate['exact_acc'] + 0.20 * aggregate['name_acc']
            + 0.20 * aggregate['valid_rate']
            - 0.05 * aggregate['malformed_rate']
            - 0.05 * aggregate['wrong_api_rate'])


## 4. Controller-Driven Sequential Training

For each stage, A and B are trained from the same selected adapter. The controller evaluates both candidates on validation families from all blocks seen so far, then carries the higher-scoring candidate into the next stage. Test families are never used in this decision.

In [ ]:
def save_adapter(model, path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    return str(path)

def release_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()

def train_candidate(parent_adapter, block_id, condition, run_dir):
    entries = partitions[block_id]['fit']
    prepared = prepare_training_entries(entries)[condition][1]
    model = load_worker(parent_adapter)
    assert len(prepared) > 0, f'No fit examples for D{block_id} policy {condition}'
    loss = train_worker(model, prepared, f'{run_dir.name}_D{block_id}_{condition}')
    adapter_path = save_adapter(model, run_dir / f'candidate_D{block_id}_{condition}')
    release_model(model)
    return adapter_path, loss

def evaluate_adapter(adapter_path, block_ids, role, condition, run_dir, stage):
    model = load_worker(adapter_path)
    metrics = {}
    for block_id in block_ids:
        entries = partitions[block_id][role]
        metrics[str(block_id)] = evaluate_entries(
            model, entries, condition,
            sample_seed=50_000 + stage * 100 + block_id
        )
    release_model(model)
    return metrics

def run_controller_on_canonical_order():
    # The controller is fitted only on the canonical order. Its policy
    # schedule is frozen before any held-out-order run begins.
    order = list(range(1, NUM_BLOCKS + 1))
    run_dir = EXPERIMENT_DIR / 'controller_canonical'
    run_dir.mkdir(parents=True, exist_ok=True)
    parent_adapter = None
    history = []
    start = time.time()

    for stage, block_id in enumerate(order, start=1):
        print(f'\n[controller] stage {stage}/{len(order)}: D{block_id}')
        candidates = {}
        for condition in ('A', 'B'):
            path, loss = train_candidate(parent_adapter, block_id, condition, run_dir)
            validation = evaluate_adapter(
                path, order[:stage], 'validation', condition, run_dir, stage
            )
            aggregate = aggregate_metrics(validation)
            candidates[condition] = {
                'adapter_path': path, 'train_loss': loss,
                'validation': validation, 'validation_aggregate': aggregate,
                'score': controller_score(aggregate),
            }
            print('  {}: score={:.4f}, exact={:.1%}, valid={:.1%}, malformed={:.1%}'.format(
                condition, candidates[condition]['score'], aggregate['exact_acc'],
                aggregate['valid_rate'], aggregate['malformed_rate']))

        # Deterministic tie-break: A. This rule is frozen before testing.
        selected = max(('A', 'B'), key=lambda c: (candidates[c]['score'], c == 'A'))
        parent_adapter = candidates[selected]['adapter_path']
        history.append({
            'stage': stage, 'block_id': block_id,
            'candidates': candidates, 'selected_policy': selected,
            'validation_only': True,
        })
        print(f'  controller selected {selected} for the next stage')

    policy_schedule = [row['selected_policy'] for row in history]
    result = {
        'order_name': 'canonical_controller_fit', 'order': order, 'seed': SEED,
        'candidate_policies': ['A', 'B'],
        'controller_score': '0.50 exact + 0.20 name + 0.20 valid - 0.05 malformed - 0.05 wrong_api',
        'token_matched': TOKEN_MATCHED,
        'policy_schedule': policy_schedule, 'history': history,
        'total_time_seconds': time.time() - start,
    }
    with open(run_dir / 'controller_results.json', 'w') as f:
        json.dump(result, f, indent=2)
    return result

def replay_frozen_schedule(order, order_name, policy_schedule):
    # Held-out stream orders never expose validation or test metrics to the
    # controller. They only replay the frozen stage-wise schedule.
    assert len(order) == len(policy_schedule) == NUM_BLOCKS
    run_dir = EXPERIMENT_DIR / order_name
    run_dir.mkdir(parents=True, exist_ok=True)
    parent_adapter = None
    training_history = []
    start = time.time()

    for stage, (block_id, condition) in enumerate(zip(order, policy_schedule), start=1):
        print(f'\n[{order_name}] replay stage {stage}/{len(order)}: D{block_id} with {condition}')
        path, loss = train_candidate(parent_adapter, block_id, condition, run_dir)
        parent_adapter = path
        training_history.append({
            'stage': stage, 'block_id': block_id, 'policy': condition,
            'validation_used': False, 'test_used': False,
            'train_loss': loss, 'adapter_path': path,
        })

    # Final evaluation is the first point at which held-out test families
    # are touched. No test metric can influence a prior training decision.
    final_condition = policy_schedule[-1]
    model = load_worker(parent_adapter)
    final_test = {}
    for block_id in order:
        final_test[str(block_id)] = evaluate_entries(
            model, partitions[block_id]['test'], final_condition,
            sample_seed=90_000 + block_id
        )
    release_model(model)
    result = {
        'order_name': order_name, 'order': order, 'seed': SEED,
        'frozen_policy_schedule': policy_schedule,
        'training_history': training_history,
        'final_test_metrics': final_test,
        'final_metrics_role': 'held-out API families only',
        'validation_used': False, 'test_used_for_selection': False,
        'total_time_seconds': time.time() - start,
    }
    with open(run_dir / 'results.json', 'w') as f:
        json.dump(result, f, indent=2)
    return result


## 5. Held-Out Stream-Order Evaluation

The controller is run without access to test metrics under each order. Report the reverse and rotated orders as held-out stream-order evaluations, alongside the canonical order for comparison.

In [ ]:
STREAM_ORDERS = {
    'canonical': list(range(1, NUM_BLOCKS + 1)),
    'reverse_heldout': list(range(NUM_BLOCKS, 0, -1)),
    'rotate_heldout': list(range(2, NUM_BLOCKS + 1)) + [1],
}

controller_result = run_controller_on_canonical_order()
policy_schedule = controller_result['policy_schedule']
all_results = {
    'canonical_test_order': replay_frozen_schedule(
        STREAM_ORDERS['canonical'], 'canonical_test', policy_schedule
    ),
    'reverse_heldout': replay_frozen_schedule(
        STREAM_ORDERS['reverse_heldout'], 'reverse_heldout', policy_schedule
    ),
    'rotate_heldout': replay_frozen_schedule(
        STREAM_ORDERS['rotate_heldout'], 'rotate_heldout', policy_schedule
    ),
}

with open(EXPERIMENT_DIR / 'results_all_orders.json', 'w') as f:
    json.dump({
        'controller_fit': controller_result,
        'held_out_order_results': all_results,
    }, f, indent=2)

print('\nFinal held-out test-family summary:')
for name, result in all_results.items():
    metrics = result['final_test_metrics'].values()
    print('{}: exact={:.1%}, name={:.1%}, valid={:.1%}'.format(
          name, np.mean([m['exact_acc'] for m in metrics]),
          np.mean([m['name_acc'] for m in metrics]), np.mean([m['valid_rate'] for m in metrics])))

## Reproducibility and interpretation

The saved `split_manifest.json` identifies the disjoint API families. Each order stores controller validation records, selected policies, adapter paths, and final test-family metrics. Do not use `final_test_metrics` to tune the controller score or selection rule. The experiment remains a component-level meta-agent study: the controller selects worker training policies, but it does not claim general self-improvement.